# 🛣️ Road Damage Enhancement Pipeline — Qt GUI

Run the cell below to launch the full GUI application.

**Requirements:** `pip install opencv-python numpy PyQt5`

In [1]:
# ── Install dependencies if needed ──────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'opencv-python', 'numpy', 'PyQt5', '-q'])

CompletedProcess(args=['C:\\Users\\KAVINDU\\anaconda3\\envs\\road_damage_project\\python.exe', '-m', 'pip', 'install', 'opencv-python', 'numpy', 'PyQt5', '-q'], returncode=0)

In [2]:
"""
Road Damage Enhancement Pipeline — Qt GUI
Gaussian Filter (3x3) → CLAHE (clip=2.0) → Unsharp Mask (k=7, a=1.0)
"""

import sys, os, csv, time
import cv2
import numpy as np

from PyQt5.QtWidgets import (
    QApplication, QMainWindow, QWidget, QLabel, QPushButton,
    QFileDialog, QSlider, QHBoxLayout, QVBoxLayout, QGridLayout,
    QGroupBox, QTabWidget, QProgressBar, QSizePolicy, QSplitter,
    QStatusBar, QMessageBox, QTextEdit, QLineEdit, QScrollArea
)
from PyQt5.QtCore import Qt, QTimer, QThread, pyqtSignal, QRect
from PyQt5.QtGui import (
    QImage, QPixmap, QPainter, QColor, QPen, QFont, QPalette
)

# ─────────────────────────────────────────────────────────
# DARK THEME
# ─────────────────────────────────────────────────────────
QSS = """
* { font-family: 'Segoe UI', Arial, sans-serif; }
QMainWindow, QWidget { background:#07090c; color:#dce4f0; font-size:12px; }
QTabWidget::pane { border:1px solid #1e2a36; background:#07090c; }
QTabBar::tab { background:#0f1419; color:#4a5872; border:1px solid #1e2a36;
               padding:9px 22px; margin-right:2px; font-size:11px; }
QTabBar::tab:selected { background:#131d28; color:#00d4aa;
                        border-bottom:2px solid #00d4aa; }
QTabBar::tab:hover { color:#dce4f0; }
QGroupBox { border:1px solid #1e2a36; border-radius:3px; margin-top:10px;
            padding-top:10px; color:#4a5872; font-size:9px; letter-spacing:2px;
            text-transform:uppercase; font-family:'Courier New'; }
QGroupBox::title { subcontrol-origin:margin; left:10px; padding:0 6px; }
QPushButton { background:#0f1419; border:1px solid #1e2a36; color:#dce4f0;
              padding:7px 16px; border-radius:2px; font-size:12px; }
QPushButton:hover { border-color:#00d4aa; color:#00d4aa; }
QPushButton:disabled { color:#1e2a36; border-color:#1e2a36; }
QPushButton#RUN { background:#00d4aa; color:#000; font-weight:bold;
                  font-size:14px; padding:11px; border:none; border-radius:2px; }
QPushButton#RUN:hover { background:#00f0c0; }
QPushButton#RUN:disabled { background:#1e2a36; color:#4a5872; }
QPushButton#CAP { background:#9d7aff; color:#000; font-weight:bold; border:none; border-radius:2px; }
QPushButton#CAP:hover { background:#b090ff; }
QPushButton#PLAY { background:#f5c518; color:#000; font-weight:bold; border:none; border-radius:2px; }
QPushButton#PLAY:hover { background:#ffd700; }
QSlider::groove:horizontal { height:5px; background:#1e2a36; border-radius:2px; }
QSlider::handle:horizontal { background:#00d4aa; width:15px; height:15px;
                              margin:-5px 0; border-radius:7px; }
QSlider::sub-page:horizontal { background:#00d4aa; border-radius:2px; }
QProgressBar { background:#1e2a36; border:none; border-radius:2px;
               height:5px; text-align:center; color:transparent; }
QProgressBar::chunk { background:#00d4aa; border-radius:2px; }
QTextEdit { background:#040608; border:1px solid #1e2a36; color:#00d4aa;
             font-family:'Courier New'; font-size:10px; border-radius:2px; }
QLineEdit { background:#0f1419; border:1px solid #1e2a36; color:#dce4f0;
             padding:5px 8px; border-radius:2px; }
QScrollBar:vertical { background:#07090c; width:6px; }
QScrollBar::handle:vertical { background:#1e2a36; border-radius:3px; min-height:20px; }
QStatusBar { background:#0f1419; color:#4a5872; font-size:10px;
             font-family:'Courier New'; }
"""

# ─────────────────────────────────────────────────────────
# PIPELINE (same as notebook)
# ─────────────────────────────────────────────────────────
def gaussian_filter(img, k=3):
    return cv2.GaussianBlur(img, (k, k), 0)

def apply_clahe(img, clip=2.0):
    c = cv2.createCLAHE(clipLimit=clip, tileGridSize=(8, 8))
    return c.apply(img)

def unsharp_mask(img, k=7, a=1.0):
    blur = cv2.GaussianBlur(img, (k, k), 0)
    return cv2.addWeighted(img, 1 + a, blur, -a, 0)

def final_pipeline(img):
    img = gaussian_filter(img, 3)
    img = apply_clahe(img, 2.0)
    img = unsharp_mask(img, 7, 1.0)
    return img

def lap_var(img):    return cv2.Laplacian(img, cv2.CV_64F).var()
def brightness(img): return float(np.mean(img))
def contrast(img):   return float(np.std(img))

# ─────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────
def gray2pix(gray):
    """Grayscale numpy → QPixmap"""
    if gray is None: return QPixmap()
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    h, w, c = rgb.shape
    return QPixmap.fromImage(QImage(rgb.data, w, h, w*c, QImage.Format_RGB888))

def bgr2pix(bgr):
    """BGR cv2 frame → QPixmap"""
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w, c = rgb.shape
    return QPixmap.fromImage(QImage(rgb.data, w, h, w*c, QImage.Format_RGB888))

def fmt_time(s):
    return f"{int(s//60)}:{int(s%60):02d}"

# ─────────────────────────────────────────────────────────
# CUSTOM WIDGETS
# ─────────────────────────────────────────────────────────
class ImgLabel(QLabel):
    """Auto-scaling image label with placeholder text."""
    def __init__(self, text='No image', parent=None):
        super().__init__(parent)
        self._pix = None
        self._ph  = text
        self.setAlignment(Qt.AlignCenter)
        self.setMinimumSize(200, 112)
        self.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.setStyleSheet('background:#000; border:1px solid #1e2a36; border-radius:2px;')

    def set_gray(self, gray):
        self._pix = gray2pix(gray) if gray is not None else None
        self.update()

    def set_pix(self, pix):
        self._pix = pix
        self.update()

    def paintEvent(self, e):
        p = QPainter(self)
        p.fillRect(self.rect(), QColor('#000'))
        if self._pix and not self._pix.isNull():
            sc = self._pix.scaled(self.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
            p.drawPixmap((self.width()-sc.width())//2, (self.height()-sc.height())//2, sc)
        else:
            p.setPen(QColor('#4a5872'))
            p.setFont(QFont('Courier New', 9))
            p.drawText(self.rect(), Qt.AlignCenter, self._ph)


class SliderCompare(QWidget):
    """Before/After split on a SINGLE canvas — no black screen."""
    def __init__(self, parent=None):
        super().__init__(parent)
        self._before = None
        self._after  = None
        self._pos    = 0.5
        self._drag   = False
        self.setMinimumSize(320, 180)
        self.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.setStyleSheet('background:#000; border:1px solid #1e2a36; border-radius:2px;')
        self.setCursor(Qt.SizeHorCursor)

    def set_images(self, before, after):
        self._before = before
        self._after  = after
        self.update()

    def mousePressEvent(self, e):
        self._drag = True
        self._pos  = max(0.0, min(1.0, e.x() / self.width()))
        self.update()

    def mouseMoveEvent(self, e):
        if self._drag:
            self._pos = max(0.0, min(1.0, e.x() / self.width()))
            self.update()

    def mouseReleaseEvent(self, e): self._drag = False

    def paintEvent(self, e):
        p = QPainter(self)
        p.fillRect(self.rect(), QColor('#000'))
        W, H = self.width(), self.height()
        split = int(self._pos * W)

        # ── draw before (full, left side) ──
        if self._before is not None:
            pix = gray2pix(self._before)
            sc  = pix.scaled(W, H, Qt.KeepAspectRatio, Qt.SmoothTransformation)
            ox  = (W - sc.width())  // 2
            oy  = (H - sc.height()) // 2
            p.save()
            p.setClipRect(0, 0, split, H)          # clip to LEFT of line
            p.drawPixmap(ox, oy, sc)
            p.restore()

        # ── draw after (right side) ──
        if self._after is not None:
            pix = gray2pix(self._after)
            sc  = pix.scaled(W, H, Qt.KeepAspectRatio, Qt.SmoothTransformation)
            ox  = (W - sc.width())  // 2
            oy  = (H - sc.height()) // 2
            p.save()
            p.setClipRect(split, 0, W - split, H)  # clip to RIGHT of line
            p.drawPixmap(ox, oy, sc)
            p.restore()

        # ── divider line + handle ──
        p.setClipping(False)
        p.setPen(QPen(QColor('#ffffff'), 2))
        p.drawLine(split, 0, split, H)
        p.setBrush(QColor('#ffffff'))
        p.setPen(Qt.NoPen)
        p.drawEllipse(split-16, H//2-16, 32, 32)
        p.setPen(QColor('#000000'))
        p.setFont(QFont('Arial', 9, QFont.Bold))
        p.drawText(QRect(split-16, H//2-16, 32, 32), Qt.AlignCenter, '⇔')

        # ── labels ──
        p.setFont(QFont('Courier New', 8))
        p.fillRect(QRect(8, 8, 72, 18), QColor(0,0,0,170))
        p.setPen(QColor('#ffffff'))
        p.drawText(QRect(8, 8, 72, 18), Qt.AlignCenter, 'ORIGINAL')
        p.fillRect(QRect(W-80, 8, 72, 18), QColor(0,212,170,60))
        p.setPen(QColor('#00d4aa'))
        p.drawText(QRect(W-80, 8, 72, 18), Qt.AlignCenter, 'ENHANCED')

        if self._before is None:
            p.setPen(QColor('#4a5872'))
            p.setFont(QFont('Courier New', 10))
            p.drawText(self.rect(), Qt.AlignCenter, 'Run pipeline first')


class HistWidget(QWidget):
    """Overlaid histograms."""
    def __init__(self, parent=None):
        super().__init__(parent)
        self._data = []
        self.setFixedHeight(90)
        self.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Fixed)
        self.setStyleSheet('background:#000; border:1px solid #1e2a36; border-radius:2px;')

    def set_data(self, data):  # [(gray, '#hex'), ...]
        self._data = data
        self.update()

    def paintEvent(self, e):
        p = QPainter(self)
        p.fillRect(self.rect(), QColor('#000'))
        W, H = self.width(), self.height()
        for gray, col in self._data:
            hist = cv2.calcHist([gray], [0], None, [256], [0,256]).flatten()
            hist /= hist.max() + 1e-9
            c = QColor(col); c.setAlpha(160)
            p.setPen(QPen(c, 1))
            bw = W / 256.0
            for i in range(256):
                bh = int(hist[i] * H)
                p.drawLine(int(i*bw), H, int(i*bw), H-bh)


class ProfileWidget(QWidget):
    """Intensity scanline profile."""
    def __init__(self, parent=None):
        super().__init__(parent)
        self._curves = []
        self.setFixedHeight(130)
        self.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Fixed)
        self.setStyleSheet('background:#000; border:1px solid #1e2a36; border-radius:2px;')

    def set_curves(self, curves):  # [(1d_arr, '#hex'), ...]
        self._curves = curves
        self.update()

    def paintEvent(self, e):
        p = QPainter(self)
        p.fillRect(self.rect(), QColor('#000'))
        W, H = self.width(), self.height()
        # grid lines
        p.setPen(QPen(QColor('#1e2a36'), 1))
        for i in range(1, 4):
            y = int(i * H / 4)
            p.drawLine(0, y, W, y)
        for curve, col in self._curves:
            n = len(curve)
            if n < 2: continue
            p.setPen(QPen(QColor(col), 1, Qt.SolidLine))
            prev = None
            for i in range(n):
                x = int(i / n * W)
                y = int((1.0 - curve[i]/255.0) * H)
                if prev: p.drawLine(prev[0], prev[1], x, y)
                prev = (x, y)


class DiffWidget(QWidget):
    """Amplified difference map."""
    def __init__(self, parent=None):
        super().__init__(parent)
        self._pix = None
        self.setMinimumSize(320, 180)
        self.setSizePolicy(QSizePolicy.Expanding, QSizePolicy.Expanding)
        self.setStyleSheet('background:#000; border:1px solid #1e2a36; border-radius:2px;')

    def set_images(self, before, after):
        if before is None or after is None:
            self._pix = None; self.update(); return
        diff = cv2.absdiff(after.astype(np.int16), before.astype(np.int16))
        amp  = np.clip(diff * 5, 0, 255).astype(np.uint8)
        rgb  = np.zeros((*amp.shape, 3), dtype=np.uint8)
        rgb[:,:,1] = amp
        rgb[:,:,2] = np.where(amp > 100, 255, 0)
        h, w, c = rgb.shape
        self._pix = QPixmap.fromImage(QImage(rgb.data, w, h, w*c, QImage.Format_RGB888))
        self.update()

    def paintEvent(self, e):
        p = QPainter(self)
        p.fillRect(self.rect(), QColor('#000'))
        if self._pix:
            sc = self._pix.scaled(self.size(), Qt.KeepAspectRatio, Qt.SmoothTransformation)
            p.drawPixmap((self.width()-sc.width())//2, (self.height()-sc.height())//2, sc)
        else:
            p.setPen(QColor('#4a5872'))
            p.setFont(QFont('Courier New', 10))
            p.drawText(self.rect(), Qt.AlignCenter, 'Run pipeline first')


# ─────────────────────────────────────────────────────────
# BATCH WORKER
# ─────────────────────────────────────────────────────────
class BatchWorker(QThread):
    progress = pyqtSignal(int, str)
    done     = pyqtSignal(list)

    def __init__(self, paths):
        super().__init__()
        self.paths = paths

    def run(self):
        res = []
        for i, path in enumerate(self.paths):
            img = cv2.imread(path, 0)
            if img is None: continue
            lb  = lap_var(img)
            out = final_pipeline(img)
            la  = lap_var(out)
            res.append({'frame': os.path.basename(path),
                        'lap_before': lb, 'lap_after': la,
                        'gain': la / max(lb, 0.001)})
            self.progress.emit(int((i+1)/len(self.paths)*100), os.path.basename(path))
        self.done.emit(res)


# ─────────────────────────────────────────────────────────
# MAIN WINDOW
# ─────────────────────────────────────────────────────────
class PipelineGUI(QMainWindow):
    def __init__(self):
        super().__init__()
        self.setWindowTitle('Road Damage Enhancement Pipeline')
        self.setMinimumSize(1280, 820)
        self.resize(1360, 900)

        # state
        self.cur_gray  = None
        self.g_orig = self.g_gauss = self.g_clahe = self.g_final = None
        self.vcap   = None
        self.vtimer = QTimer(); self.vtimer.timeout.connect(self._vtick)
        self.vplay  = False
        self.vtotal = 0
        self._cur_frame = None

        self._build()
        self.setStyleSheet(QSS)
        self._log('// Road Damage Enhancement Pipeline ready')
        self._log('// Open an image or video → Run Pipeline')

    # ── BUILD ────────────────────────────────────────────
    def _build(self):
        cw = QWidget(); self.setCentralWidget(cw)
        root = QVBoxLayout(cw)
        root.setContentsMargins(0, 0, 0, 0)
        root.setSpacing(0)

        # HEADER
        hdr = QWidget()
        hdr.setFixedHeight(60)
        hdr.setStyleSheet('background:#0f1419; border-bottom:1px solid #1e2a36;')
        hl = QHBoxLayout(hdr)
        hl.setContentsMargins(20, 0, 20, 0)
        title = QLabel('Road Damage  <span style="color:#00d4aa">Enhancement Pipeline</span>')
        title.setTextFormat(Qt.RichText)
        title.setFont(QFont('Segoe UI', 17, QFont.Bold))
        sub = QLabel('Gaussian(3×3) → CLAHE(2.0) → Unsharp(k=7,a=1.0)  ·  General Pipeline  ·  Best Laplacian Variance')
        sub.setFont(QFont('Courier New', 9))
        sub.setStyleSheet('color:#4a5872')
        vv = QVBoxLayout(); vv.setSpacing(1)
        vv.addWidget(title); vv.addWidget(sub)
        hl.addLayout(vv); hl.addStretch()
        root.addWidget(hdr)

        # TABS
        body = QWidget(); body.setContentsMargins(12, 10, 12, 10)
        bl = QVBoxLayout(body); bl.setContentsMargins(0,0,0,0)
        self.tabs = QTabWidget()
        self.tabs.addTab(self._tab_input(),    '📂  Input & Video')
        self.tabs.addTab(self._tab_pipeline(), '⚡  Pipeline Stages')
        self.tabs.addTab(self._tab_compare(),  '🔍  Before / After')
        self.tabs.addTab(self._tab_analysis(), '📊  Analysis')
        self.tabs.addTab(self._tab_batch(),    '🗂  Batch Export')
        bl.addWidget(self.tabs)
        root.addWidget(body)

        # STATUS BAR
        self.sb = QStatusBar(); self.setStatusBar(self.sb)
        self.sb.showMessage('Ready')

    # ── TAB 1: INPUT ─────────────────────────────────────
    def _tab_input(self):
        w = QWidget()
        l = QVBoxLayout(w); l.setSpacing(10)

        # Open buttons
        br = QHBoxLayout()
        for label, slot in [('🖼  Open Image', self._open_img),
                             ('🎬  Open Video', self._open_vid)]:
            b = QPushButton(label); b.setFixedHeight(38); b.clicked.connect(slot)
            br.addWidget(b)
        br.addStretch(); l.addLayout(br)

        self.inp_info = QLabel('No file loaded')
        self.inp_info.setFont(QFont('Courier New', 9))
        self.inp_info.setStyleSheet('color:#4a5872')
        l.addWidget(self.inp_info)

        # Split: preview | video controls
        sp = QSplitter(Qt.Horizontal)

        # left: preview
        pbox = QGroupBox('PREVIEW / VIDEO')
        pl = QVBoxLayout(pbox)
        self.inp_prev = ImgLabel('No file loaded')
        self.inp_prev.setMinimumHeight(320)
        pl.addWidget(self.inp_prev)
        sp.addWidget(pbox)

        # right: video controls
        vbox = QGroupBox('VIDEO CONTROLS')
        vl = QVBoxLayout(vbox); vl.setSpacing(8)

        vl.addWidget(QLabel('🎬  Scrub Timeline:'))
        self.vslider = QSlider(Qt.Horizontal)
        self.vslider.setRange(0, 1000)
        self.vslider.sliderMoved.connect(self._vseek)
        vl.addWidget(self.vslider)

        self.vtlbl = QLabel('0:00 / 0:00')
        self.vtlbl.setFont(QFont('Courier New', 10))
        self.vtlbl.setStyleSheet('color:#f5c518')
        vl.addWidget(self.vtlbl)

        row1 = QHBoxLayout()
        self.vplaybtn = QPushButton('▶  Play')
        self.vplaybtn.setObjectName('PLAY')
        self.vplaybtn.setFixedHeight(36)
        self.vplaybtn.clicked.connect(self._vtoggle)
        row1.addWidget(self.vplaybtn)
        for txt, s in [('⏮ −10s',-10),('◀◀ −5s',-5),('+5s ▶▶',5),('+10s ⏭',10)]:
            b = QPushButton(txt); b.setFixedHeight(34)
            b.clicked.connect(lambda _, sec=s: self._vskip(sec))
            row1.addWidget(b)
        vl.addLayout(row1)

        # Frame preview
        vl.addWidget(QLabel('📸  Current Frame:'))
        self.frame_prev = ImgLabel('No frame')
        self.frame_prev.setFixedHeight(130)
        vl.addWidget(self.frame_prev)

        self.cap_info = QLabel('')
        self.cap_info.setFont(QFont('Courier New', 9))
        self.cap_info.setStyleSheet('color:#00d4aa')
        vl.addWidget(self.cap_info)

        capbtn = QPushButton('📸  Capture This Frame  →  Use as Input')
        capbtn.setObjectName('CAP'); capbtn.setFixedHeight(38)
        capbtn.clicked.connect(self._capture)
        vl.addWidget(capbtn)

        vl.addStretch()
        sp.addWidget(vbox)
        sp.setSizes([720, 360])
        l.addWidget(sp)

        # RUN button
        self.runbtn = QPushButton('⚡  RUN ENHANCEMENT PIPELINE')
        self.runbtn.setObjectName('RUN'); self.runbtn.setFixedHeight(50)
        self.runbtn.setEnabled(False); self.runbtn.clicked.connect(self._run)
        l.addWidget(self.runbtn)
        return w

    # ── TAB 2: PIPELINE ──────────────────────────────────
    def _tab_pipeline(self):
        w = QWidget()
        l = QVBoxLayout(w); l.setSpacing(10)

        # 4 stage cards
        sr = QHBoxLayout(); sr.setSpacing(8)
        stages = [
            ('orig',  'Stage 01 · Original',     '#4a5872', 'Input frame'),
            ('gauss', 'Stage 02 · Gaussian 3×3',  '#00d4aa', 'Noise removal'),
            ('clahe', 'Stage 03 · CLAHE 2.0',     '#9d7aff', 'Local contrast'),
            ('final', 'Stage 04 · Unsharp Mask',  '#ff6b2b', 'Edge sharpening'),
        ]
        self.stg_img = {}; self.stg_lap = {}; self.stg_pb = {}
        for key, title, color, desc in stages:
            box = QGroupBox(title)
            box.setStyleSheet(f'QGroupBox {{ border-top:3px solid {color}; }}')
            bl = QVBoxLayout(box)
            img = ImgLabel('waiting…'); img.setMinimumHeight(160)
            bl.addWidget(img)
            d = QLabel(desc)
            d.setFont(QFont('Courier New', 8))
            d.setStyleSheet('color:#4a5872'); bl.addWidget(d)
            lap = QLabel('Laplacian: —')
            lap.setFont(QFont('Courier New', 9))
            lap.setStyleSheet(f'color:{color}'); bl.addWidget(lap)
            pb = QProgressBar(); pb.setValue(0); bl.addWidget(pb)
            self.stg_img[key] = img
            self.stg_lap[key] = lap
            self.stg_pb[key]  = pb
            sr.addWidget(box)
        l.addLayout(sr)

        # Metrics
        mbox = QGroupBox('QUALITY METRICS')
        mg = QGridLayout(mbox)
        self.met = {}
        metrics = [
            ('lap_o',  'Laplacian Before',  '#4a5872'),
            ('lap_f',  'Laplacian After',   '#00d4aa'),
            ('gain',   'Gain Factor',       '#f5c518'),
            ('pct',    'Sharpness Gain',    '#6be06b'),
            ('bright', 'Brightness After',  '#9d7aff'),
            ('cont',   'Contrast After',    '#ff6b2b'),
        ]
        for i, (k, lbl, col) in enumerate(metrics):
            l2 = QLabel(lbl); l2.setFont(QFont('Courier New', 8))
            l2.setStyleSheet('color:#4a5872')
            v = QLabel('—'); v.setFont(QFont('Segoe UI', 20, QFont.Bold))
            v.setStyleSheet(f'color:{col}')
            mg.addWidget(l2, 0, i); mg.addWidget(v, 1, i)
            self.met[k] = v
        l.addWidget(mbox)

        # Log
        lgbox = QGroupBox('PROCESSING LOG')
        ll = QVBoxLayout(lgbox)
        self.logw = QTextEdit()
        self.logw.setReadOnly(True); self.logw.setFixedHeight(110)
        ll.addWidget(self.logw)
        l.addWidget(lgbox)
        return w

    # ── TAB 3: COMPARE ───────────────────────────────────
    def _tab_compare(self):
        w = QWidget()
        l = QVBoxLayout(w); l.setSpacing(10)

        # mode buttons
        mr = QHBoxLayout()
        ml = QLabel('View Mode:')
        ml.setFont(QFont('Courier New', 9))
        ml.setStyleSheet('color:#4a5872')
        mr.addWidget(ml)
        self.mbtns = []
        for i, nm in enumerate(['⇔  Slider', '⧉  Side by Side', '🟡  Diff Map']):
            b = QPushButton(nm); b.setCheckable(True); b.setChecked(i==0)
            b.setFixedHeight(32)
            b.clicked.connect(lambda _, idx=i: self._set_mode(idx))
            self.mbtns.append(b); mr.addWidget(b)
        mr.addStretch(); l.addLayout(mr)

        # stacked
        self.cmp_stack = QTabWidget()
        self.cmp_stack.tabBar().setVisible(False)

        # slider
        sw = QWidget(); sl = QVBoxLayout(sw); sl.setContentsMargins(0,0,0,0)
        self.slcmp = SliderCompare(); sl.addWidget(self.slcmp)
        self.cmp_stack.addTab(sw, 's')

        # side by side
        sbsw = QWidget(); sbsl = QHBoxLayout(sbsw); sbsl.setSpacing(8)
        def lbld(txt, img, col):
            b = QWidget(); bl = QVBoxLayout(b); bl.setContentsMargins(0,0,0,0)
            lbl = QLabel(txt); lbl.setFont(QFont('Courier New', 9))
            lbl.setStyleSheet(f'color:{col}; padding:2px 0;')
            bl.addWidget(lbl); bl.addWidget(img); return b
        self.sbs_o = ImgLabel('Original')
        self.sbs_f = ImgLabel('Enhanced')
        sbsl.addWidget(lbld('ORIGINAL', self.sbs_o, '#4a5872'))
        sbsl.addWidget(lbld('ENHANCED', self.sbs_f, '#00d4aa'))
        self.cmp_stack.addTab(sbsw, 'sbs')

        # diff
        dw = QWidget(); dl = QVBoxLayout(dw)
        dl2 = QLabel('DIFFERENCE MAP  —  bright = most changed pixels')
        dl2.setFont(QFont('Courier New', 9)); dl2.setStyleSheet('color:#4a5872')
        dl.addWidget(dl2)
        self.diffwgt = DiffWidget(); dl.addWidget(self.diffwgt)
        self.cmp_stack.addTab(dw, 'd')

        l.addWidget(self.cmp_stack)
        return w

    # ── TAB 4: ANALYSIS ──────────────────────────────────
    def _tab_analysis(self):
        w = QWidget()
        l = QVBoxLayout(w); l.setSpacing(10)

        hbox = QGroupBox('PIXEL HISTOGRAMS  —  Original / After CLAHE / Final')
        hl = QVBoxLayout(hbox)
        self.histw = HistWidget(); hl.addWidget(self.histw)
        leg = QHBoxLayout()
        for col, txt in [('#4a5872','■ Original'),('#9d7aff','■ After CLAHE'),('#00d4aa','■ Final')]:
            lb = QLabel(txt); lb.setFont(QFont('Courier New', 9))
            lb.setStyleSheet(f'color:{col}'); leg.addWidget(lb)
        leg.addStretch(); hl.addLayout(leg)
        l.addWidget(hbox)

        pbox = QGroupBox('INTENSITY PROFILE  —  horizontal midline scanline')
        pl = QVBoxLayout(pbox)
        self.profw = ProfileWidget(); pl.addWidget(self.profw)
        leg2 = QHBoxLayout()
        for col, txt in [('#4a5872','── Original'),('#9d7aff','── After CLAHE'),('#00d4aa','── Final')]:
            lb = QLabel(txt); lb.setFont(QFont('Courier New', 9))
            lb.setStyleSheet(f'color:{col}'); leg2.addWidget(lb)
        leg2.addStretch(); pl.addLayout(leg2)
        l.addWidget(pbox)

        sbox = QGroupBox('ENHANCEMENT SUMMARY')
        sl = QVBoxLayout(sbox)
        self.sumw = QTextEdit()
        self.sumw.setReadOnly(True); self.sumw.setFixedHeight(150)
        sl.addWidget(self.sumw)
        l.addWidget(sbox)
        return w

    # ── TAB 5: BATCH ─────────────────────────────────────
    def _tab_batch(self):
        w = QWidget()
        l = QVBoxLayout(w); l.setSpacing(10)

        fr = QHBoxLayout()
        self.bpath = QLineEdit(); self.bpath.setPlaceholderText('Folder containing images…')
        bb = QPushButton('Browse…'); bb.clicked.connect(self._bbrowse)
        fr.addWidget(self.bpath); fr.addWidget(bb); l.addLayout(fr)

        cr = QHBoxLayout()
        self.cpath = QLineEdit(); self.cpath.setPlaceholderText('Output CSV path…')
        cb = QPushButton('Save As…'); cb.clicked.connect(self._cbrowse)
        cr.addWidget(self.cpath); cr.addWidget(cb); l.addLayout(cr)

        rb = QPushButton('⚡  Run Batch Pipeline  →  Export CSV')
        rb.setObjectName('RUN'); rb.setFixedHeight(44); rb.clicked.connect(self._brun)
        l.addWidget(rb)

        self.bprog = QProgressBar(); self.bprog.setValue(0); l.addWidget(self.bprog)
        self.bstat = QLabel('Ready')
        self.bstat.setFont(QFont('Courier New', 9))
        self.bstat.setStyleSheet('color:#4a5872'); l.addWidget(self.bstat)

        rbox = QGroupBox('RESULTS')
        rl = QVBoxLayout(rbox)
        self.bres = QTextEdit(); self.bres.setReadOnly(True)
        self.bres.setFont(QFont('Courier New', 9))
        rl.addWidget(self.bres)
        l.addWidget(rbox)
        return w

    # ─── FILE / VIDEO OPERATIONS ─────────────────────────
    def _open_img(self):
        path, _ = QFileDialog.getOpenFileName(self, 'Open Image', '',
                   'Images (*.jpg *.jpeg *.png *.bmp *.tif *.tiff)')
        if not path: return
        gray = cv2.imread(path, 0)
        if gray is None: QMessageBox.warning(self,'Error','Could not read image.'); return
        self.cur_gray = gray
        self.inp_prev.set_gray(gray)
        self.frame_prev.set_gray(gray)
        self.inp_info.setText(f'Image: {os.path.basename(path)}  |  {gray.shape[1]}×{gray.shape[0]} px')
        self.runbtn.setEnabled(True)
        self._log(f'Loaded: {os.path.basename(path)}  ({gray.shape[1]}×{gray.shape[0]})')
        self.sb.showMessage(f'Loaded: {os.path.basename(path)}')

    def _open_vid(self):
        path, _ = QFileDialog.getOpenFileName(self, 'Open Video', '',
                   'Videos (*.mp4 *.avi *.mov *.mkv *.wmv)')
        if not path: return
        if self.vcap: self.vcap.release()
        self.vcap = cv2.VideoCapture(path)
        if not self.vcap.isOpened():
            QMessageBox.warning(self,'Error','Could not open video.'); return
        self.vtotal = int(self.vcap.get(cv2.CAP_PROP_FRAME_COUNT))
        fps = self.vcap.get(cv2.CAP_PROP_FPS)
        dur = self.vtotal / max(fps,1)
        W   = int(self.vcap.get(cv2.CAP_PROP_FRAME_WIDTH))
        H   = int(self.vcap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.inp_info.setText(
            f'Video: {os.path.basename(path)}  |  {W}×{H}  |  {fps:.1f}fps  |  {dur:.1f}s')
        self.vslider.setValue(0); self._vseek(0)
        self._log(f'Video loaded: {os.path.basename(path)}  ({W}×{H}, {fps:.1f}fps, {dur:.1f}s)')
        self.sb.showMessage(f'Video: {os.path.basename(path)}')

    def _vtick(self):
        if not self.vcap: return
        ret, frame = self.vcap.read()
        if not ret:
            self.vtimer.stop(); self.vplay=False
            self.vplaybtn.setText('▶  Play'); return
        fn  = int(self.vcap.get(cv2.CAP_PROP_POS_FRAMES))
        fps = self.vcap.get(cv2.CAP_PROP_FPS)
        if self.vtotal>0: self.vslider.setValue(int(fn/self.vtotal*1000))
        self._show_frame(frame, fn, fps)

    def _vtoggle(self):
        if not self.vcap: return
        if self.vplay:
            self.vtimer.stop(); self.vplay=False
            self.vplaybtn.setText('▶  Play')
        else:
            fps = self.vcap.get(cv2.CAP_PROP_FPS)
            self.vtimer.start(max(1, int(1000/max(fps,1))))
            self.vplay=True; self.vplaybtn.setText('⏸  Pause')

    def _vseek(self, val):
        if not self.vcap: return
        fn = int(val/1000*self.vtotal)
        self.vcap.set(cv2.CAP_PROP_POS_FRAMES, fn)
        ret, frame = self.vcap.read()
        if ret: self._show_frame(frame, fn, self.vcap.get(cv2.CAP_PROP_FPS))

    def _vskip(self, secs):
        if not self.vcap: return
        fps = self.vcap.get(cv2.CAP_PROP_FPS)
        cur = self.vcap.get(cv2.CAP_PROP_POS_FRAMES)
        new = max(0, min(self.vtotal-1, cur + secs*fps))
        self.vcap.set(cv2.CAP_PROP_POS_FRAMES, new)
        ret, frame = self.vcap.read()
        if ret: self._show_frame(frame, int(new), fps)

    def _show_frame(self, frame, fn, fps):
        self._cur_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        self.inp_prev.set_pix(bgr2pix(frame))
        self.frame_prev.set_pix(bgr2pix(frame))
        t   = fn / max(fps,1)
        tot = self.vtotal / max(fps,1)
        self.vtlbl.setText(f'{fmt_time(t)}  /  {fmt_time(tot)}  ·  frame {fn}')

    def _capture(self):
        if self._cur_frame is None:
            QMessageBox.information(self,'Info','Open a video and scrub to a frame first.'); return
        self.cur_gray = self._cur_frame.copy()
        self.runbtn.setEnabled(True)
        fps = self.vcap.get(cv2.CAP_PROP_FPS) if self.vcap else 1
        fn  = int(self.vcap.get(cv2.CAP_PROP_POS_FRAMES)) if self.vcap else 0
        t   = fn / max(fps,1)
        self.cap_info.setText(
            f'✓ Captured  {self.cur_gray.shape[1]}×{self.cur_gray.shape[0]}  @  {t:.2f}s')
        self._log(f'Frame captured: {self.cur_gray.shape[1]}×{self.cur_gray.shape[0]} @ {t:.2f}s')

    # ─── PIPELINE ────────────────────────────────────────
    def _run(self):
        if self.cur_gray is None: return
        self.runbtn.setEnabled(False)
        self.runbtn.setText('⏳  Processing…')
        QApplication.processEvents()

        img = self.cur_gray; t0 = time.time()
        self._log('═══ Pipeline started ═══')

        for k in self.stg_pb: self.stg_pb[k].setValue(0)

        # orig
        self.stg_img['orig'].set_gray(img)
        self.stg_lap['orig'].setText(f'Laplacian: {lap_var(img):.2f}')
        self.stg_pb['orig'].setValue(100); QApplication.processEvents()

        # stage 1: gaussian
        self._log('Stage 1 → Gaussian Blur (3×3)…', 's1')
        g1 = gaussian_filter(img, 3)
        self.stg_img['gauss'].set_gray(g1)
        self.stg_lap['gauss'].setText(f'Laplacian: {lap_var(g1):.2f}')
        self.stg_pb['gauss'].setValue(100)
        self._log('Stage 1 ✓', 'ok'); QApplication.processEvents()

        # stage 2: CLAHE
        self._log('Stage 2 → CLAHE (clip=2.0, tile=8×8)…', 's2')
        g2 = apply_clahe(g1, 2.0)
        self.stg_img['clahe'].set_gray(g2)
        self.stg_lap['clahe'].setText(f'Laplacian: {lap_var(g2):.2f}')
        self.stg_pb['clahe'].setValue(100)
        self._log('Stage 2 ✓', 'ok'); QApplication.processEvents()

        # stage 3: unsharp
        self._log('Stage 3 → Unsharp Mask (k=7, a=1.0)…', 's3')
        g3 = unsharp_mask(g2, 7, 1.0)
        self.stg_img['final'].set_gray(g3)
        self.stg_lap['final'].setText(f'Laplacian: {lap_var(g3):.2f}')
        self.stg_pb['final'].setValue(100)
        self._log('Stage 3 ✓', 'ok'); QApplication.processEvents()

        self.g_orig=img; self.g_gauss=g1; self.g_clahe=g2; self.g_final=g3

        # metrics
        lO  = lap_var(img); lF = lap_var(g3)
        bF  = brightness(g3); cF = contrast(g3)
        gain = lF / max(lO, 0.001)
        pct  = (lF-lO) / max(lO,0.001) * 100
        self.met['lap_o'].setText(f'{lO:.1f}')
        self.met['lap_f'].setText(f'{lF:.1f}')
        self.met['gain'].setText(f'{gain:.3f}×')
        self.met['pct'].setText(f'+{pct:.1f}%')
        self.met['bright'].setText(f'{bF:.1f}')
        self.met['cont'].setText(f'{cF:.1f}')

        # compare
        self.slcmp.set_images(img, g3)
        self.sbs_o.set_gray(img); self.sbs_f.set_gray(g3)
        self.diffwgt.set_images(img, g3)

        # analysis
        self.histw.set_data([(img,'#4a5872'),(g2,'#9d7aff'),(g3,'#00d4aa')])
        row = img.shape[0]//2
        self.profw.set_curves([
            (img[row,:].astype(float),'#4a5872'),
            (g2[row,:].astype(float),'#9d7aff'),
            (g3[row,:].astype(float),'#00d4aa'),
        ])

        # summary
        elapsed = time.time()-t0
        self.sumw.setPlainText(
            f"Pipeline : Gaussian(3×3) → CLAHE(clip=2.0,tile=8×8) → Unsharp(k=7,a=1.0)\n"
            f"{'─'*54}\n"
            f"Size      : {img.shape[1]} × {img.shape[0]} px\n"
            f"Sharpness : {lO:.2f}  →  {lF:.2f}   (+{pct:.1f}%)\n"
            f"Brightness: {brightness(img):.1f}  →  {bF:.1f}\n"
            f"Contrast  : {contrast(img):.1f}  →  {cF:.1f}\n"
            f"Gain      : {gain:.4f}×\n"
            f"Time      : {elapsed:.3f}s\n"
            f"{'─'*54}\n"
            f"Result    : Sharpness improved by {pct:.1f}%\n"
            f"            Validates General Pipeline selection."
        )

        ms = int((time.time()-t0)*1000)
        self._log(f'═══ Done — Lap {lO:.1f}→{lF:.1f} (+{pct:.1f}%) · gain {gain:.3f}× · {ms}ms ═══','ok')
        self.runbtn.setEnabled(True)
        self.runbtn.setText('⚡  RUN ENHANCEMENT PIPELINE')
        self.sb.showMessage(f'Done — sharpness +{pct:.1f}%  gain {gain:.3f}×  ({ms}ms)')
        self.tabs.setCurrentIndex(1)  # jump to Pipeline tab

    # ─── COMPARE MODE ────────────────────────────────────
    def _set_mode(self, idx):
        for i,b in enumerate(self.mbtns): b.setChecked(i==idx)
        self.cmp_stack.setCurrentIndex(idx)

    # ─── BATCH ───────────────────────────────────────────
    def _bbrowse(self):
        f = QFileDialog.getExistingDirectory(self,'Select Image Folder')
        if f: self.bpath.setText(f)

    def _cbrowse(self):
        p,_ = QFileDialog.getSaveFileName(self,'Save CSV','','CSV (*.csv)')
        if p: self.cpath.setText(p)

    def _brun(self):
        folder = self.bpath.text()
        if not folder or not os.path.isdir(folder):
            QMessageBox.warning(self,'Error','Select a valid folder.'); return
        paths = [os.path.join(folder,f) for f in os.listdir(folder)
                 if f.lower().endswith(('.jpg','.jpeg','.png','.bmp'))]
        if not paths:
            QMessageBox.information(self,'Info','No images found.'); return
        self.bres.clear()
        self._bworker = BatchWorker(paths)
        self._bworker.progress.connect(lambda p,n: (
            self.bprog.setValue(p), self.bstat.setText(f'Processing: {n}  ({p}%)')))
        self._bworker.done.connect(self._bdone)
        self._bworker.start()

    def _bdone(self, res):
        self.bstat.setText(f'Done — {len(res)} frames')
        lines = [f'{"Frame":<40} {"Lap Before":>12} {"Lap After":>12} {"Gain":>8}']
        lines.append('─'*76)
        for r in res:
            lines.append(f'{r["frame"]:<40} {r["lap_before"]:>12.2f} {r["lap_after"]:>12.2f} {r["gain"]:>8.3f}×')
        self.bres.setPlainText('\n'.join(lines))
        csv_out = self.cpath.text()
        if csv_out:
            with open(csv_out,'w',newline='') as f:
                w = csv.writer(f)
                w.writerow(['frame','lap_before','lap_after','gain'])
                for r in res: w.writerow([r['frame'],r['lap_before'],r['lap_after'],r['gain']])
            self._log(f'CSV saved: {csv_out}','ok')

    # ─── LOG ─────────────────────────────────────────────
    _COLS = {'ok':'#6be06b','s1':'#00d4aa','s2':'#9d7aff','s3':'#ff6b2b'}

    def _log(self, msg, t='i'):
        ts  = time.strftime('%H:%M:%S')
        col = self._COLS.get(t,'#00d4aa')
        self.logw.append(
            f'<span style="color:#4a5872">{ts}</span> '
            f'<span style="color:{col}">{msg}</span>')

    def closeEvent(self, e):
        if self.vcap: self.vcap.release()
        super().closeEvent(e)


# ─────────────────────────────────────────────────────────
# LAUNCH  (works in Jupyter + standalone)
# ─────────────────────────────────────────────────────────
app = QApplication.instance() or QApplication(sys.argv)
app.setStyle('Fusion')
win = PipelineGUI()
win.show()
# Keep event loop alive in notebook
try:
    from IPython import get_ipython
    ip = get_ipython()
    if ip:
        ip.run_line_magic('gui', 'qt5')
    else:
        app.exec_()
except Exception:
    app.exec_()


Request for "qt5" will be ignored because `QT_API` environment variable is set to "pyside6"
